# Constitution Evidence — Kaggle GPU experiment

Enable **GPU** and **Internet** in Kaggle. This notebook uses only the French corpus. It runs an experimental local Qwen candidate and saves raw responses; it does not create human legal judgments or a final benchmark. Use a published `develop` branch or upload this working tree first.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json, hashlib, time
ROOT = Path('/kaggle/working/moroccan-constitution-rag')
if not ROOT.exists():
    subprocess.run(['git','clone','--branch','develop','https://github.com/ShalomJunior/moroccan-constitution-rag.git',str(ROOT)],check=True)
os.chdir(ROOT)
print(subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True).stdout.strip())
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout.strip())


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[dense]','huggingface_hub'],check=True)
subprocess.run(['apt-get','update','-qq'],check=True,stdout=subprocess.DEVNULL)
subprocess.run(['apt-get','install','-y','-qq','poppler-utils','tesseract-ocr','tesseract-ocr-fra'],check=True,stdout=subprocess.DEVNULL)
(ROOT/'data/raw').mkdir(parents=True,exist_ok=True)
pdf = ROOT/'data/raw/constitution_2011_fr_marocma.pdf'
if not pdf.exists():
    subprocess.run(['curl','-L','--fail','-o',str(pdf),'https://maroc.ma/sites/default/files/2024-10/constitution_du_1er_juillet_2011.pdf'],check=True)
assert hashlib.sha256(pdf.read_bytes()).hexdigest() == '4101d45b4733a6b36eee77bd9ef4175b23160103089b003b13907e8b6d59333d'
# Tesseract's installed French pack is used instead of downloading a second copy.
from constitution_evidence.ingestion import ingest
from constitution_evidence.store import read_jsonl
import shutil
language_dir = Path('/usr/share/tesseract-ocr/5/tessdata')
manifest = ingest(ROOT/'configs/corpus.json',root=ROOT,tessdata=language_dir)
print(manifest['article_count'],manifest['passage_count'],manifest['corpus_hash'])


In [ ]:
from huggingface_hub import HfApi, hf_hub_download
repo = 'Qwen/Qwen3-4B-GGUF'
revision = HfApi().model_info(repo).sha
model_path = hf_hub_download(repo_id=repo,filename='Qwen3-4B-Q4_K_M.gguf',revision=revision)
digest = hashlib.sha256()
with open(model_path, 'rb') as stream:
    for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
        digest.update(block)
model_hash = digest.hexdigest()
print(revision,model_hash)


In [ ]:
source = Path('/kaggle/working/llama.cpp')
if not source.exists(): subprocess.run(['git','clone','--depth','1','https://github.com/ggml-org/llama.cpp.git',str(source)],check=True)
if not shutil.which('nvcc'): raise RuntimeError('CUDA compiler unavailable in this Kaggle image; select a GPU image with CUDA toolkit')
build = Path('/kaggle/working/llama-build')
subprocess.run(['cmake','-S',str(source),'-B',str(build),'-DGGML_CUDA=ON','-DCMAKE_BUILD_TYPE=Release'],check=True)
subprocess.run(['cmake','--build',str(build),'-j','4','--target','llama-server'],check=True)
server = subprocess.Popen([str(build/'bin/llama-server'),'-m',model_path,'--alias','Qwen3-4B','--host','127.0.0.1','--port','8080','-c','4096','-ngl','99','--jinja'],stdout=open('/kaggle/working/llama-server.log','w'),stderr=subprocess.STDOUT)
import urllib.request
for _ in range(180):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8080/health',timeout=1) as response:
            if response.status == 200: break
    except Exception: pass
    time.sleep(2)
else: raise RuntimeError('llama-server did not become healthy; inspect /kaggle/working/llama-server.log')
print('llama-server ready')


In [ ]:
from constitution_evidence.retrieval import BM25
from constitution_evidence.evidence import pack
from constitution_evidence.local_generation import LocalGenerator
passages = read_jsonl(ROOT/'data/processed/passages.jsonl')
questions = read_jsonl(ROOT/'data/annotations/calibration.jsonl')[:3]
engine = BM25(passages)
generator = LocalGenerator(model='Qwen3-4B',timeout=300)
run_dir = ROOT/'runs/kaggle-qwen4b-calibration'
run_dir.mkdir(parents=True,exist_ok=True)
rows=[]
for q in questions:
    try:
        answer = generator.generate(q['question'],pack(engine.search(q['question'],5),7000),manifest)
        rows.append({'question_id':q['question_id'],'answer':answer})
    except Exception as exc:
        rows.append({'question_id':q['question_id'],'error':str(exc)})
with (run_dir/'responses.jsonl').open('w') as f:
    for row in rows: f.write(json.dumps(row,ensure_ascii=False)+'\n')
(run_dir/'manifest.json').write_text(json.dumps({'model_repo':repo,'model_revision':revision,'model_sha256':model_hash,'corpus_hash':manifest['corpus_hash'],'commit':subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True).stdout.strip(),'llama_commit':subprocess.run(['git','-C',str(source),'rev-parse','HEAD'],capture_output=True,text=True).stdout.strip(),'review_status':manifest['review_status'],'question_count':len(questions)},indent=2))
print(rows)


Review `runs/kaggle-qwen4b-calibration/` and the source PDF manually. These three calibration questions are provisional. A full evaluation requires the 180 family French benchmark and independent judgments.
